In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

file_path = (
    project_root / "data" / "external"
    / "Coffee Shop Sales.xlsx"
)

processed_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_excel(
    file_path,
    sheet_name="Transactions"
)

df_clean = df_raw.copy()

print("Jumlah data awal:", len(df_clean))
print("Jumlah kolom:", len(df_clean.columns))

display(df_clean.head())

Jumlah data awal: 149116
Jumlah kolom: 11


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [2]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print("Nama kolom setelah standardisasi:")
print(df_clean.columns.tolist())

Nama kolom setelah standardisasi:
['transaction_id', 'transaction_date', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail']


In [3]:
text_columns = [
    "store_location",
    "product_category",
    "product_type",
    "product_detail"
]

for col in text_columns:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
    )

print("Standardisasi teks selesai.")

Standardisasi teks selesai.


In [4]:
from datetime import time

df_clean["transaction_date"] = pd.to_datetime(
    df_clean["transaction_date"],
    errors="coerce"
)

def format_time(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, time):
        return value.strftime("%H:%M:%S")

    parsed = pd.to_datetime(
        str(value),
        format="%H:%M:%S",
        errors="coerce"
    )

    if pd.isna(parsed):
        return pd.NA

    return parsed.strftime("%H:%M:%S")

df_clean["transaction_time"] = (
    df_clean["transaction_time"].apply(format_time)
)

print("Tanggal kosong:",
      df_clean["transaction_date"].isna().sum())

print("Waktu kosong:",
      df_clean["transaction_time"].isna().sum())

display(
    df_clean[
        ["transaction_date", "transaction_time"]
    ].head()
)

Tanggal kosong: 0
Waktu kosong: 0


,transaction_date,transaction_time
0,2023-01-01,07:06:11
1,2023-01-01,07:08:56
2,2023-01-01,07:14:04
3,2023-01-01,07:20:24
4,2023-01-01,07:22:41


In [5]:
df_clean["total_amount"] = (
    df_clean["transaction_qty"]
    * df_clean["unit_price"]
).round(2)

print("=== STATISTIK NILAI TRANSAKSI ===")

display(df_clean["total_amount"].describe())

print(
    "Nilai transaksi nol/negatif:",
    (df_clean["total_amount"] <= 0).sum()
)

=== STATISTIK NILAI TRANSAKSI ===


count    149116.000000
mean          4.686367
std           4.227099
min           0.800000
25%           3.000000
50%           3.750000
75%           6.000000
max         360.000000
Name: total_amount, dtype: float64

Nilai transaksi nol/negatif: 0


In [6]:
category_mapping = {
    "Coffee": "beverage",
    "Tea": "beverage",
    "Drinking Chocolate": "beverage",

    "Bakery": "food",
    "Packaged Chocolate": "food",

    "Coffee beans": "retail",
    "Loose Tea": "retail",

    "Branded": "merchandise",
    "Flavours": "other"
}

df_clean["category_group"] = (
    df_clean["product_category"].map(category_mapping)
)

print("=== DISTRIBUSI KATEGORI UMUM ===")

display(
    df_clean["category_group"]
    .value_counts(dropna=False)
)

print(
    "Kategori belum dipetakan:",
    df_clean["category_group"].isna().sum()
)

=== DISTRIBUSI KATEGORI UMUM ===


category_group
beverage       115333
food            23283
other            6790
retail           2963
merchandise       747
Name: count, dtype: int64

Kategori belum dipetakan: 0


In [7]:
df_clean["data_type"] = "public"
df_clean["source_dataset"] = "Coffee Shop Sales"
df_clean["source_platform"] = "Kaggle"

print("=== METADATA DATASET ===")

display(
    df_clean[
        ["data_type", "source_dataset", "source_platform"]
    ].head()
)

=== METADATA DATASET ===


,data_type,source_dataset,source_platform
0,public,Coffee Shop Sales,Kaggle
1,public,Coffee Shop Sales,Kaggle
2,public,Coffee Shop Sales,Kaggle
3,public,Coffee Shop Sales,Kaggle
4,public,Coffee Shop Sales,Kaggle


In [8]:
print("=== VALIDASI AKHIR SALES CLEANING ===")

assert len(df_clean) == len(df_raw), \
    "Jumlah record berubah!"

assert df_clean["transaction_id"].is_unique, \
    "Transaction ID tidak unik!"

assert df_clean["transaction_date"].notna().all(), \
    "Ada tanggal tidak valid!"

assert df_clean["transaction_time"].notna().all(), \
    "Ada waktu tidak valid!"

assert (df_clean["transaction_qty"] > 0).all(), \
    "Ada kuantitas tidak valid!"

assert (df_clean["unit_price"] > 0).all(), \
    "Ada harga tidak valid!"

assert df_clean["category_group"].notna().all(), \
    "Ada kategori belum dipetakan!"

assert df_clean["data_type"].eq("public").all(), \
    "Metadata jenis data tidak sesuai!"

print("Jumlah record:", len(df_clean))
print("Jumlah kategori umum:", df_clean["category_group"].nunique())
print("Duplikasi transaction_id:",
      df_clean["transaction_id"].duplicated().sum())

print("\nValidasi berhasil!")

=== VALIDASI AKHIR SALES CLEANING ===
Jumlah record: 149116
Jumlah kategori umum: 5
Duplikasi transaction_id: 0

Validasi berhasil!


In [9]:
output_path = (
    processed_dir / "coffee_shop_sales_clean.csv"
)

df_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset berhasil disimpan:")
print(output_path)

Dataset berhasil disimpan:
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\data\processed\coffee_shop_sales_clean.csv


In [10]:
cleaning_log = pd.DataFrame([
    {
        "step": 1,
        "process": "Standardisasi nama kolom",
        "description": "Memastikan nama kolom konsisten"
    },
    {
        "step": 2,
        "process": "Standardisasi teks",
        "description": "Menghapus spasi awal dan akhir"
    },
    {
        "step": 3,
        "process": "Standardisasi tanggal dan waktu",
        "description": "Mengubah tanggal dan waktu ke format analisis"
    },
    {
        "step": 4,
        "process": "Feature engineering",
        "description": "Menambahkan total_amount dari kuantitas dan harga"
    },
    {
        "step": 5,
        "process": "Pengelompokan kategori",
        "description": "Memetakan kategori publik ke kelompok analisis"
    },
    {
        "step": 6,
        "process": "Metadata",
        "description": "Menambahkan label public dan identitas sumber"
    },
    {
        "step": 7,
        "process": "Validasi",
        "description": "Memeriksa jumlah record, harga, kuantitas, dan kategori"
    }
])

log_path = reports_dir / "sales_cleaning_log.csv"

cleaning_log.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

display(cleaning_log)

print("Cleaning log tersimpan:", log_path)

,step,process,description
0,1,Standardisasi nama kolom,Memastikan nama kolom konsisten
1,2,Standardisasi teks,Menghapus spasi awal dan akhir
2,3,Standardisasi tanggal dan waktu,Mengubah tanggal dan waktu ke format analisis
3,4,Feature engineering,Menambahkan total_amount dari kuantitas dan harga
4,5,Pengelompokan kategori,Memetakan kategori publik ke kelompok analisis
5,6,Metadata,Menambahkan label public dan identitas sumber
6,7,Validasi,"Memeriksa jumlah record, harga, kuantitas, dan..."


Cleaning log tersimpan: a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\reports\sales_cleaning_log.csv
